# Module 10 lab · Decompositions

The code from every lesson of [this module](https://aicourses.dev/mathematics/decompositions), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/mathematics/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 10.2 A new ruler: basis and coordinates

The same point has different numbers under a different pair of arrows, and with perpendicular unit arrows the numbers are dot products. A new ruler is the first step to taking a matrix apart.

[Read the lesson](https://aicourses.dev/mathematics/decompositions/basis-and-coordinates)

### Perpendicular unit arrows

In [ ]:
import numpy as np

u1 = np.array([0.6, 0.8])
u2 = np.array([-0.8, 0.6])
x = np.array([7.0, 1.0])
print((x @ u1).round(4), (x @ u2).round(4))
U = np.column_stack([u1, u2])
c = U.T @ x                        # all the dot products at once
print(c.round(4), (U @ c).round(4))
B = np.column_stack([[1.0, 0.0], [1.0, 1.0]])   # slanted arrows
p = np.array([3.0, 2.0])
print(np.linalg.solve(B, p), B.T @ p)

*The lesson shows:*

```text
5.0 -5.0
[ 5. -5.] [7. 1.]
[1. 2.] [3. 5.]
```

### A better ruler for the digits

Do the same in 64 directions. Take neighbouring pixel pairs: pixels 0 and 1, 2 and 3, and so on, 32 pairs in all. For each pair make two arrows of length 1. One is “their sum”, with $1/\sqrt{2} \approx 0.7071$ on both pixels. The other is “their difference”, with $0.7071$ on the first and $-0.7071$ on the second. All 64 arrows are perpendicular to one another, so the 64 by 64 matrix $\mathbf{Q}$ with these arrows as rows is orthogonal, and the coordinates of a digit are `Q @ x`. This is a simpler cousin of what JPEG does.

In [ ]:
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16                      # pixel values 0 to 1
order = rng.permutation(len(X))  # one seeded shuffle
X, y = X[order], y[order]
Xtr, ytr = X[:1297], y[:1297]    # 1,297 to learn from
Xte, yte = X[1297:], y[1297:]    # 500 held out
Q = np.zeros((64, 64))
for i in range(32):
    Q[i, 2 * i] = Q[i, 2 * i + 1] = 2 ** -0.5          # the sum arrow
    Q[32 + i, 2 * i], Q[32 + i, 2 * i + 1] = 2 ** -0.5, -(2 ** -0.5)
print(np.allclose(Q @ Q.T, np.eye(64)))
z = Q @ Xte[0]
print(np.linalg.norm(Xte[0]).round(3), np.linalg.norm(z).round(3))
Z = Xtr @ Q.T                      # every training digit in the new ruler
print(((Z[:, :32] ** 2).sum() / (Z ** 2).sum()).round(3))

*The lesson shows:*

```text
True
4.01 4.01
0.84
```

## 10.3 The closest point: projection and least squares

No line goes through three points, but one is closest, and the miss is perpendicular to it. This is projection from Module 02 turned into least squares.

[Read the lesson](https://aicourses.dev/mathematics/decompositions/least-squares)

### The miss is at a right angle

In [ ]:
import numpy as np

A = np.array([[1.0, 0], [1, 1], [1, 2]])
y = np.array([1.0, 2, 2])
c = np.linalg.solve(A.T @ A, A.T @ y)       # the normal equations
miss = y - A @ c
print(c.round(4), miss.round(4), (miss ** 2).sum().round(4))
print((A.T @ miss).round(10) + 0.0)
c2 = np.linalg.lstsq(A, y, rcond=None)[0]   # the same, in one call
print(c2.round(4))
y2 = np.array([3.0, 1, 2])
print(np.linalg.lstsq(A, y2, rcond=None)[0].round(4))

*The lesson shows:*

```text
[1.1667 0.5   ] [-0.1667  0.3333 -0.1667] 0.1667
[0. 0.]
[1.1667 0.5   ]
[ 2.5 -0.5]
```

### The same line on the penguins

$$
b = \frac{46350 \times 1819 + 3 \times 77350}{21819} \approx 3874.73, \qquad
w = \frac{12 \times 77350 + 3 \times 46350}{21819} \approx 48.9138
$$

In [ ]:
flipper = np.array([181, 183, 185, 192, 193, 193, 197, 198, 205, 208, 216, 222], dtype=float)  # mm
mass = np.array([3625, 3550, 3150, 2700, 2925, 3650, 3750, 4400, 4300, 4200, 4750, 5350], dtype=float)  # g
P = np.column_stack([np.ones(12), flipper - 198])
b, w = np.linalg.lstsq(P, mass, rcond=None)[0]
print(round(b, 2), round(w, 4))
print(np.polyfit(flipper, mass, 1).round(4))

*The lesson shows:*

```text
3874.73 48.9138
[   48.9138 -5810.2021]
```

### Where this lives in AI

Least squares is the fit behind the squared-error loss of Module 00. For the digit classifier of Module 03 it gives something more: the best possible matrix for squared error, in one step. Make the table $\mathbf{A}$ from the 1,297 training digits with a column of ones for the shift, make the target a table of ten one-hot columns (1 for the right digit, 0 for the others), and `lstsq` solves all ten fits at once:

In [ ]:
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16                      # pixel values 0 to 1
order = rng.permutation(len(X))  # one seeded shuffle
X, y = X[order], y[order]
Xtr, ytr = X[:1297], y[:1297]    # 1,297 to learn from
Xte, yte = X[1297:], y[1297:]    # 500 held out
A = np.hstack([Xtr, np.ones((1297, 1))])
Y = np.eye(10)[ytr]
P, _, rank, _ = np.linalg.lstsq(A, Y, rcond=None)
print(rank, ((A @ P - Y) ** 2).sum(axis=1).mean().round(5))
Ate = np.hstack([Xte, np.ones((500, 1))])
print(((Ate @ P).argmax(axis=1) == yte).mean())

*The lesson shows:*

```text
62 0.30474
0.944
```

## 10.4 Directions that do not turn: eigenvectors

Some arrows only stretch when a matrix moves them, and the stretch is the eigenvalue. A symmetric matrix has perpendicular ones, which leads to the SVD.

[Read the lesson](https://aicourses.dev/mathematics/decompositions/eigenvectors)

### Finding one by multiplying again and again

In [ ]:
import numpy as np

S = np.array([[2.0, 1.0], [1.0, 2.0]])
v = np.array([1.0, 0.0])
for step in range(1, 7):
    v = S @ v
    v = v / np.linalg.norm(v)          # shrink back to length 1
    print(step, v.round(4), (v @ S @ v).round(4))

*The lesson shows:*

```text
1 [0.8944 0.4472] 2.8
2 [0.7809 0.6247] 2.9756
3 [0.7328 0.6805] 2.9973
4 [0.7158 0.6983] 2.9997
5 [0.71   0.7042] 3.0
6 [0.7081 0.7061] 3.0
```

### Symmetric means perpendicular

The two eigenvectors of $\mathbf{S}$, $(1, 1)$ and $(1, -1)$, are perpendicular: their dot product is $1 - 1 = 0$. For $\mathbf{T}$ too: $(2, 1) \cdot (1, -2) = 0$. That is not luck. A real matrix that equals its own transpose, a **symmetric** matrix, always has perpendicular eigenvectors for different eigenvalues, and a perpendicular set to pick from if two share an eigenvalue. This can be proved; here you check it. The condition matters. Take $\mathbf{M}$ with rows $(2, 1)$ and $(0, 3)$, which is not symmetric. It has the eigenvector $(1, 0)$ with $\lambda = 2$ and $(1, 1)$ with $\lambda = 3$, and their directions are not at a right angle.

In [ ]:
vals, vecs = np.linalg.eigh(S)           # eigh is for symmetric matrices
print(vals, vecs.round(4), sep="\n")
M = np.array([[2.0, 1.0], [0.0, 3.0]])
vals, vecs = np.linalg.eig(M)            # eig works for any square matrix
print(vals, vecs.round(4), (vecs[:, 0] @ vecs[:, 1]).round(4), sep="\n")

*The lesson shows:*

```text
[1. 3.]
[[-0.7071  0.7071]
 [ 0.7071  0.7071]]
[2. 3.]
[[1.     0.7071]
 [0.     0.7071]]
0.7071
```

### Where this lives in AI

In [ ]:
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16                      # pixel values 0 to 1
order = rng.permutation(len(X))  # one seeded shuffle
X, y = X[order], y[order]
Xtr, ytr = X[:1297], y[:1297]    # 1,297 to learn from
Xte, yte = X[1297:], y[1297:]    # 500 held out
A = np.hstack([Xtr, np.ones((1297, 1))])    # 64 pixels and a column of ones
H = 2 / 1297 * A.T @ A                      # the table of curvatures
v = np.ones(65)
for _ in range(300):
    v = H @ v
    v = v / np.linalg.norm(v)
lam = v @ H @ v
print(lam.round(4), np.linalg.eigvalsh(H)[-1].round(4), (2 / lam).round(5))

*The lesson shows:*

```text
22.8967 22.8967 0.08735
```

Power iteration on this 65 by 65 table finds the largest curvature, 22.8967, and `eigvalsh` (the eigenvalues of a symmetric matrix, increasing) agrees. The limit is $2 / 22.8967 = 0.08735$. Now train the matrix from zeros at the two rates Module 06’s project tried, 0.0865 and 0.0917, which are 99% and 105% of that limit, with the squared-error update of Module 06:

In [ ]:
Y = np.eye(10)[ytr]
for lr in (0.0865, 0.0917):
    P = np.zeros((65, 10))
    for _ in range(500):
        P = P - lr * 2 / 1297 * A.T @ (A @ P - Y)
    print(lr, ((A @ P - Y) ** 2).sum(axis=1).mean())

*The lesson shows:*

```text
0.0865 0.314021183376635
0.0917 1.7437364929232182e+40
```

## 10.5 Rotate, stretch, rotate: the SVD

Every matrix, whatever its shape, is a rotation, a stretch along the axes and another rotation. The SVD is the centre of the module and the tool the next lessons use.

[Read the lesson](https://aicourses.dev/mathematics/decompositions/the-svd)

### Finding the stretches by hand

In [ ]:
import numpy as np

A = np.array([[1.0, 1.0], [0.0, 1.0]])
U, s, Vt = np.linalg.svd(A)
print(s.round(4), s.prod().round(4), (s ** 2).sum(), (A ** 2).sum())
v1, u1 = np.abs(Vt[0]), np.abs(U[:, 0])
print(np.degrees(np.arctan2(v1[1], v1[0])).round(1),
      np.degrees(np.arctan2(u1[1], u1[0])).round(1))
print((U @ np.diag(s) @ Vt).round(4))
print(np.linalg.svd(np.array([[3.0, 0], [4, 0]]))[1].round(4))

*The lesson shows:*

```text
[1.618 0.618] 1.0 3.0 3.0
58.3 31.7
[[1. 1.]
 [0. 1.]]
[5. 0.]
```

### Any shape in NumPy

Nothing so far needed $\mathbf{A}$ to be square. A matrix with $m$ rows and $n$ columns has $\min(m, n)$ singular values, and the “economy” form keeps only the pieces that matter: $\mathbf{U}$ is $m$ by $\min(m, n)$, `s` has $\min(m, n)$ numbers and $\mathbf{V}^\top$ is $\min(m, n)$ by $n$. Try it on the ten-template matrix of the digit classifier from the Module 03 project, 10 rows by 64 columns:

In [ ]:
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16                      # pixel values 0 to 1
order = rng.permutation(len(X))  # one seeded shuffle
X, y = X[order], y[order]
Xtr, ytr = X[:1297], y[:1297]    # 1,297 to learn from
Xte, yte = X[1297:], y[1297:]    # 500 held out
W = np.array([Xtr[ytr == k].mean(axis=0) for k in range(10)])
U, s, Vt = np.linalg.svd(W, full_matrices=False)
print(U.shape, s.shape, Vt.shape)
print(s.round(3))
print(np.allclose(U @ np.diag(s) @ Vt, W), np.allclose(Vt @ Vt.T, np.eye(10)))
print(np.linalg.matrix_rank(W), (s ** 2).sum().round(3), (W ** 2).sum().round(3))

*The lesson shows:*

```text
(10, 10) (10,) (10, 64)
[10.167  2.412  2.111  1.8    1.334  1.279  1.061  0.884  0.696  0.571]
True True
10 123.027 123.027
```

## 10.6 Keep the big ones: low-rank approximation

Drop the small stretches and a matrix barely changes, and nothing of that rank is closer. You rebuild an 8 by 8 digit from one, two and four pieces.

[Read the lesson](https://aicourses.dev/mathematics/decompositions/low-rank)

### One digit as a matrix

A digit of the course is 64 numbers, and 64 numbers can be laid out as an 8 by 8 matrix, which has an SVD of its own. Take test digit 0, which is a 1:

In [ ]:
import numpy as np
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16                      # pixel values 0 to 1
order = rng.permutation(len(X))  # one seeded shuffle
X, y = X[order], y[order]
Xtr, ytr = X[:1297], y[:1297]    # 1,297 to learn from
Xte, yte = X[1297:], y[1297:]    # 500 held out
M = Xte[0].reshape(8, 8)
U, s, Vt = np.linalg.svd(M)
print(s.round(3))

*The lesson shows:*

```text
[3.734 1.331 0.442 0.322 0.218 0.139 0.    0.   ]
```

Now rebuild from the first $r$ pieces, and compare the error with the formula from the dropped stretches:

In [ ]:
for r in (1, 2, 4, 6):
    Mr = (U[:, :r] * s[:r]) @ Vt[:r]
    err = np.linalg.norm(M - Mr)
    print(r, err.round(3), (err / np.linalg.norm(M)).round(3),
          np.sqrt((s[r:] ** 2).sum()).round(3))
ranks = [np.linalg.matrix_rank(m.reshape(8, 8)) for m in Xte]
print(M[:, [0, 7]].sum(), np.bincount(ranks))

*The lesson shows:*

```text
1 1.462 0.365 1.462
2 0.605 0.151 0.605
4 0.259 0.065 0.259
6 0.0 0.0 0.0
0.0 [  0   0   0   3  15  65 373  44]
```

### The whole table of digits

Now stack all 1,297 training digits as the rows of a table. First **centre** it: subtract the average digit from every row, so that what is left is how each digit differs from the typical one. The table $\mathbf{C}$ is 1,297 by 64, and the SVD splits it into 64 pieces, each a direction in the 64-pixel space with a stretch. Its squared size, the sum of squares of all entries, is the sum of $\sigma_i^2$, so the share that the first $k$ pieces keep is $(\sigma_1^2 + \cdots + \sigma_k^2)$ divided by all of them.

In [ ]:
mean = Xtr.mean(axis=0)
C = Xtr - mean                    # each row: a digit minus the average digit
Uc, sc, Vc = np.linalg.svd(C, full_matrices=False)
share = sc ** 2 / (sc ** 2).sum()
for k in (2, 5, 10, 20):
    print(k, share[:k].sum().round(4))
for k in (0, 2, 5, 10, 20):
    D = (Xte - mean) @ Vc[:k].T @ Vc[:k]      # test digits from k directions
    print(k, ((Xte - mean - D) ** 2).sum(axis=1).mean().round(3))

*The lesson shows:*

```text
2 0.2864
5 0.5448
10 0.7385
20 0.8953
0 4.716
2 3.397
5 2.157
10 1.254
20 0.525
```

## 10.7 Covariance and PCA

A table of how every pair of pixels moves together has top directions that are the best axes for the cloud. PCA puts the digits in two dimensions and ties together the SVD and Module 08.

[Read the lesson](https://aicourses.dev/mathematics/decompositions/covariance-and-pca)

### The best axes are eigenvectors

The best axis keeps the most spread, and it also leaves the shortest dashed segments: what is kept and what is lost add up to the total, so more of one is less of the other. This is the same fact as Eckart and Young’s in lesson 6, seen from the cloud. The centred table $\mathbf{X}_c$ has an SVD, and its right singular vectors are the eigenvectors of $\mathbf{C}$, with $\mathbf{C}$’s eigenvalues equal to $\sigma_i^2 / (N - 1)$. So the share kept by the first $k$ directions is also $(\sigma_1^2 + \cdots + \sigma_k^2)$ over the sum of all the $\sigma_i^2$, the same as in lesson 6. If the centred table had the stretches 4, 3, 2 and 1, the squares are 16, 9, 4 and 1, they add to 30, and the first two directions keep $25/30 = 0.833$.

In [ ]:
import numpy as np

flipper = np.array([181, 183, 185, 192, 193, 193, 197, 198, 205, 208, 216, 222], dtype=float)  # mm
mass = np.array([3625, 3550, 3150, 2700, 2925, 3650, 3750, 4400, 4300, 4200, 4750, 5350], dtype=float)  # g
fd, md = flipper - flipper.mean(), mass - mass.mean()
print((fd * md).sum() / 11, np.cov(flipper, mass)[0, 1])
P = np.array([[1, 1], [2, 3], [3, 2], [4, 4.]])
Pc = P - P.mean(axis=0)
Cov = Pc.T @ Pc / 3
vals, vecs = np.linalg.eigh(Cov)
print(Cov.round(4), vals.round(4), sep="\n")
print(np.abs(vecs[:, 1]).round(4), np.abs(Pc @ vecs[:, 1]).round(3))

*The lesson shows:*

```text
8085.2272727273 8085.2272727273
[[1.6667 1.3333]
 [1.3333 1.6667]]
[0.3333 3.    ]
[0.7071 0.7071] [2.121 0.    0.   2.121]
```

### The digits in two dimensions

Do it to the digits. The pixel at row 3, column 3 and its right-hand neighbour (counting from 0, as code does) should move together, since a stroke often passes both. Row 5, column 3 and row 6, column 5, which a stroke seldom joins, should move against each other.

In [ ]:
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16                      # pixel values 0 to 1
order = rng.permutation(len(X))  # one seeded shuffle
X, y = X[order], y[order]
Xtr, ytr = X[:1297], y[:1297]    # 1,297 to learn from
Xte, yte = X[1297:], y[1297:]    # 500 held out
mean = Xtr.mean(axis=0)
C = Xtr - mean
Cov = C.T @ C / (len(C) - 1)
print(Cov.shape, np.allclose(Cov, np.cov(Xtr, rowvar=False)))
print(Cov[27, 28].round(4), Cov[43, 53].round(4), np.trace(Cov).round(4))

*The lesson shows:*

```text
(64, 64) True
0.0616 -0.0683 4.6896
```

Now the eigenvectors of this table, biggest eigenvalue first, and a look at the digits in the top two:

In [ ]:
vals, vecs = np.linalg.eigh(Cov)
vals, vecs = vals[::-1], vecs[:, ::-1]        # biggest first
print(vals[:4].round(3), (vals[:2].sum() / vals.sum()).round(4))
Ztr, Zte = C @ vecs[:, :2], (Xte - mean) @ vecs[:, :2]
d = ((Zte[:, None, :] - Ztr[None]) ** 2).sum(axis=2)
print((ytr[d.argmin(axis=1)] == yte).mean())

*The lesson shows:*

```text
[0.702 0.641 0.55  0.392] 0.2864
0.578
```

## 10.8 Low rank inside a model: LoRA

LoRA freezes a big matrix and learns two thin ones whose product is the change. You count the parameters it saves, which is why a low rank matters in practice.

[Read the lesson](https://aicourses.dev/mathematics/decompositions/lora)

### Counting the saved numbers

For GPT-3 175B the paper’s setting is $r = 4$, adapting the query and value matrices of every layer (Section 4.2). Those matrices are 12,288 by 12,288, as wide as the model (Brown et al., 2020, Table 2.1 gives GPT-3’s width as 12,288 and its depth as 96 layers; the LoRA paper writes that their full rank can be as high as 12,288). So one such matrix has $12288^2$ numbers, and LoRA trains $4 \times (12288 + 12288) = 98{,}304$ for it. The code below counts both.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
m, n, r = 4096, 4096, 8
print(m * n, r * (m + n), m * n // (r * (m + n)))
d, r = 12288, 4
print(d * d, r * (d + d), d * d // (r * (d + d)))
print(96 * 2 * r * (d + d), round(175e9 / (96 * 2 * r * (d + d))))

*The lesson shows:*

```text
16777216 65536 256
150994944 98304 1536
18874368 9272
```

### Why a low rank might be enough

Two facts connect this to the SVD. First, the product $\mathbf{B}\mathbf{A}$ has rank at most $r$, because each of its columns is a mix of the $r$ columns of $\mathbf{B}$. Second, every matrix of rank $r$ can be written this way: from its SVD take $\mathbf{B} = \mathbf{U}_r\boldsymbol{\Sigma}_r$ and $\mathbf{A} = \mathbf{V}_r^\top$. So LoRA searches exactly the low-rank changes, and by Eckart–Young the best rank-$r$ stand-in for a given change $\Delta\mathbf{W}$ is its truncated SVD. First check that a zero $\mathbf{B}$ leaves the output alone, and that the two thin multiplies agree with the big one:

In [ ]:
W0 = rng.normal(size=(10, 64))            # a frozen matrix
B = np.zeros((10, 2))                     # starts at zero
A = rng.normal(0, 0.1, size=(2, 64))      # starts random
x = rng.normal(size=64)
print(np.allclose(W0 @ x + B @ (A @ x), W0 @ x))
B = rng.normal(size=(10, 2))              # as if trained: any numbers
print(np.linalg.matrix_rank(B @ A), np.allclose(B @ (A @ x), (B @ A) @ x))

*The lesson shows:*

```text
True
2 True
```

At the start the adapted layer equals the frozen one (`True`). With any $\mathbf{B}$, the product has rank 2, and computing $\mathbf{B}(\mathbf{A}\mathbf{x})$ gives the same answer as $(\mathbf{B}\mathbf{A})\mathbf{x}$ without ever building the big table: two thin multiplies are cheap. Then take a table you know, the ten-template matrix, and write its best rank 2 as two thin tables:

In [ ]:
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16                      # pixel values 0 to 1
order = rng.permutation(len(X))  # one seeded shuffle
X, y = X[order], y[order]
Xtr, ytr = X[:1297], y[:1297]    # 1,297 to learn from
W = np.array([Xtr[ytr == k].mean(axis=0) for k in range(10)])
U, s, Vt = np.linalg.svd(W, full_matrices=False)
B, A = U[:, :2] * s[:2], Vt[:2]
print(B.shape, A.shape, W.size, B.size + A.size)
print(np.linalg.matrix_rank(B @ A), np.linalg.norm(W - B @ A).round(3),
      np.sqrt((s[2:] ** 2).sum()).round(3))

*The lesson shows:*

```text
(10, 2) (2, 64) 640 148
2 3.719 3.719
```

## 10.9 Project: compress the digits and a weight matrix

You compress the digit images and a trained weight matrix, plot error against rank and choose a rank you can defend with accuracy. It uses every tool in the module on real matrices.

[Read the lesson](https://aicourses.dev/mathematics/decompositions/project-compress)

**This is the project.** These cells are starting points from the lesson; the steps between them are yours to write. A cell that uses a name from one of your steps stops with a NameError until you have written that step.

### Part 1: the digits

Centre the 1,297 training digits, take the SVD, and project every digit onto its first $k$ directions: coordinates are `(x - mean) @ Vt[:k].T`, from lesson 2. Then classify each held-out digit by the nearest training digit in that smaller space, as in Module 02.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16                      # pixel values 0 to 1
order = rng.permutation(len(X))  # one seeded shuffle
X, y = X[order], y[order]
Xtr, ytr = X[:1297], y[:1297]    # 1,297 to learn from
Xte, yte = X[1297:], y[1297:]    # 500 held out
mean = Xtr.mean(axis=0)
C = Xtr - mean
Vt = np.linalg.svd(C, full_matrices=False)[2]
def nearest(Ztr, Zte):
    d = ((Zte[:, None, :] - Ztr[None]) ** 2).sum(axis=2)
    return (ytr[d.argmin(axis=1)] == yte).mean()
for k in (2, 5, 10, 20):
    print(k, nearest(C @ Vt[:k].T, (Xte - mean) @ Vt[:k].T))
print("all 64", nearest(Xtr, Xte))

*The lesson shows:*

```text
2 0.578
5 0.932
10 0.976
20 0.988
all 64 0.992
```

Two directions give 0.578, five give 0.932, ten give 0.976 and twenty give 0.988, against 0.992 for all 64. Accuracy climbs far faster than variance did: lesson 6 found that 5 directions keep only 0.5448 of the squared size, yet 0.932 of the digits are read right. Now the error curve for the held-out digits:

In [ ]:
import matplotlib.pyplot as plt

Xc = Xte - mean
err = [((Xc - Xc @ Vt[:k].T @ Vt[:k]) ** 2).sum(axis=1).mean() for k in range(65)]
plt.plot(err)
plt.xlabel("directions kept")
plt.ylabel("mean squared error per digit")
plt.show()
print(round(err[0], 3), round(err[20], 3), round(err[64], 6))

*The lesson shows:*

```text
4.716 0.525 0.0
```

### The trained matrix

Retrain Module 09’s classifier: softmax with cross-entropy, from zeros, plain gradient descent with a rate of 0.5 for 2,000 steps. Then take the SVD of its 10 by 64 weight matrix and rebuild it at each rank, keeping the shift whole.

In [ ]:
def softmax(S):
    E = np.exp(S - S.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)
A = np.hstack([Xtr, np.ones((1297, 1))])
Y = np.eye(10)[ytr]
Q = np.zeros((65, 10))
for _ in range(2000):
    Q = Q - 0.5 * A.T @ (softmax(A @ Q) - Y) / len(A)
Wt, b = Q[:64].T, Q[64]                   # 10 by 64 weights, and the shift
print(((Xte @ Wt.T + b).argmax(axis=1) == yte).mean())
print(np.abs(Wt.sum(axis=0)).max() < 1e-9)
Uw, sw, Vw = np.linalg.svd(Wt, full_matrices=False)
print(sw.round(3))
for r in range(1, 10):
    Wr = (Uw[:, :r] * sw[:r]) @ Vw[:r]
    acc = ((Xte @ Wr.T + b).argmax(axis=1) == yte).mean()
    print(r, acc, ((sw[:r] ** 2).sum() / (sw ** 2).sum()).round(3))

*The lesson shows:*

```text
0.982
True
[9.802 8.675 7.877 7.772 7.157 6.636 4.546 4.319 4.011 0.   ]
1 0.194 0.216
2 0.364 0.385
3 0.624 0.525
4 0.748 0.661
5 0.89 0.776
6 0.932 0.875
7 0.948 0.922
8 0.98 0.964
9 0.982 1.0
```

### Module 03’s templates, returned to

In lesson 03.9 you squashed the ten-template matrix to rank 1 by projecting on the average template, and it answered 8 for every image. Now the SVD gives the best rank 1 and rank 5.

In [ ]:
T = np.array([Xtr[ytr == k].mean(axis=0) for k in range(10)])
Ut, st, Vtt = np.linalg.svd(T, full_matrices=False)
for r in (1, 5, 10):
    Tr = (Ut[:, :r] * st[:r]) @ Vtt[:r]
    p = (Xte @ Tr.T).argmax(axis=1)
    print(r, (p == yte).mean(), np.unique(p).size)

*The lesson shows:*

```text
1 0.084 1
5 0.796 10
10 0.9 10
```